# Chronos 2 — Load


In [ ]:
# !pip install -r requirements/chronos2.txt


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_ROOT=Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT/"requirements").exists():
    REPO_ROOT=REPO_ROOT.parent
ROOT=REPO_ROOT/"load"
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))

from chronos import BaseChronosPipeline
from preprocessing import load_data, describe_data, SITES, CONTEXT_LENGTH, PREDICTION_LENGTH
from src.metrics import print_metrics

BASE_QUANTILES=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
TAIL_QUANTILES=[0.01,0.025,0.05,0.95,0.975,0.99]
OUTPUT_QUANTILES=sorted(set(BASE_QUANTILES+TAIL_QUANTILES))
QUANTILES=BASE_QUANTILES
MODEL_QUANTILES=[q for q in OUTPUT_QUANTILES if q not in (0.025,0.975)]
OUTPUT_DIR=ROOT/"outputs"/"predictions"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
data=load_data(ROOT/"data")
print(describe_data(data))


## GEFCom2014-L


In [ ]:
# GEFCom2014-L (cleaned input; no extra cleaning)
from gefcom import load_gefcom_load, load_initial_and_tasks, evaluate_gefcom_load, LOAD_WEATHER
gef_load=load_gefcom_load(ROOT/"data"/"gefcom_load_data.csv")
gef_context, gef_tasks=load_initial_and_tasks(gef_load)
GEF_OUTPUT=ROOT/"outputs"/"predictions"; GEF_OUTPUT.mkdir(parents=True,exist_ok=True)


In [ ]:
GEF_VARIANT="with_covariates"
gef_pipeline=BaseChronosPipeline.from_pretrained("amazon/chronos-2",device_map="cuda")
rolling=gef_context.copy(); frames=[]
for task_id,future in tqdm(gef_tasks):
    context=rolling.sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy(); cols=["item_id","timestamp","target",*LOAD_WEATHER] if GEF_VARIANT=="with_covariates" else ["item_id","timestamp","target"]
    pred=gef_pipeline.predict_df(context[cols],prediction_length=len(future),quantile_levels=QUANTILES,id_column="item_id",timestamp_column="timestamp",target="target")
    pred["task_id"]=task_id; frames.append(pred); rolling=pd.concat([rolling,future],ignore_index=True)
gef_result=pd.concat(frames,ignore_index=True).merge(gef_load[["item_id","timestamp","target","task_id"]],on=["item_id","timestamp","task_id"],how="inner")
if "prediction" not in gef_result and "0.5" in gef_result: gef_result["prediction"]=gef_result["0.5"]
gef_result.to_csv(GEF_OUTPUT/f"chronos2_gefcom_load_{GEF_VARIANT}.csv",index=False)
gef_summary=evaluate_gefcom_load(gef_result,QUANTILES); display(gef_summary[["task_id","raw_aql","normalized_aql"]])
gef_summary.to_csv(GEF_OUTPUT/f"chronos2_gefcom_load_summary.csv",index=False)


## PG&E competition


In [ ]:
VARIANT="with_covariates"  # or "without_covariates"
DEVICE="cuda"
pipeline=BaseChronosPipeline.from_pretrained("amazon/chronos-2",device_map=DEVICE)
FUTURE_COLS=data.covariates
PAST_COLS=[]


In [ ]:
rolling={s:data.context_by_site[s].copy() for s in data.context_by_site}
frames=[]
for i in tqdm(range(len(next(iter(data.daily_by_site.values()))))):
    day={s:data.daily_by_site[s][i][1].sort_values("timestamp").iloc[:PREDICTION_LENGTH].copy() for s in data.daily_by_site}
    for site,future in day.items():
        context=rolling[site].sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
        kwargs=dict(prediction_length=PREDICTION_LENGTH,quantile_levels=MODEL_QUANTILES,id_column="item_id",timestamp_column="timestamp",target="target")
        if VARIANT=="with_covariates":
            future_df=future[["item_id","timestamp",*FUTURE_COLS]].copy()
            pred=pipeline.predict_df(context,future_df=future_df,**kwargs)
        else:
            pred=pipeline.predict_df(context[["item_id","timestamp","target"]],**kwargs)
        frames.append(pred)
    for site in day:
        rolling[site]=pd.concat([rolling[site],day[site]],ignore_index=True).sort_values("timestamp").iloc[-CONTEXT_LENGTH:].copy()
result=pd.concat(frames,ignore_index=True)
if "predictions" in result.columns: result=result.rename(columns={"predictions":"prediction"})
if "prediction" not in result.columns: result["prediction"]=result["0.5"]
result=result.merge(data.test[["item_id","timestamp","target"]],on=["item_id","timestamp"],how="inner")

# Keep a common 15-quantile CSV schema. Unsupported model quantiles remain NaN.
for _q in OUTPUT_QUANTILES:
    if str(_q) not in result.columns:
        result[str(_q)] = np.nan
result.to_csv(OUTPUT_DIR/f"chronos2_{VARIANT}.csv",index=False)
print_metrics(result,QUANTILES,label=f"chronos2_{VARIANT}")
